# 02B — Clinical signal structure


In [15]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import fisher_exact, chi2_contingency, mannwhitneyu, kruskal, spearmanr, ks_2samp, entropy, norm
from IPython.display import display

PROCESSED = Path('../../data/processed')
df = pd.read_csv(PROCESSED / 'DMD_variants_annotated.csv')

print('Loaded:', PROCESSED / 'DMD_variants_annotated.csv')
print('Shape:', df.shape)


Loaded: ../../data/processed/DMD_variants_annotated.csv
Shape: (11308, 30)


In [16]:
import sys
from pathlib import Path

PROJECT_ROOT = None
for rel in ('../..', '..', '.'):
    cand = (Path.cwd() / rel).resolve()
    if (cand / 'src' / 'utils.py').exists():
        PROJECT_ROOT = cand
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('project root with src/utils.py not found')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import (
    add_consistency_flags,
    bh_adjust,
    chi2_table,
    ecdf_xy,
    fisher_bool,
    frame_mismatch,
    kruskal_group,
    mann_whitney_bool,
    mut_cons_mismatch,
    odds_ratio_ci,
    path_cons_mismatch,
    prepare_eda_dataframe,
    spearman_xy,
)

d = prepare_eda_dataframe(df)

print('Prepared shape:', d.shape)


Prepared shape: (11308, 59)


## 1. 📊 Pathogenic fraction by phenotype


In [17]:
tbl = d.groupby('phenotype').agg(pathogenic_fraction=('pathogenic', 'mean'), n=('var_id', 'size'), pathogenic_n=('pathogenic', 'sum')).reset_index().sort_values('n', ascending=False)
display(tbl)
fig = px.bar(tbl, x='phenotype', y='pathogenic_fraction', hover_data=['n', 'pathogenic_n'], title='Pathogenic fraction by phenotype')
fig.show()


,phenotype,pathogenic_fraction,n,pathogenic_n
1,DMD,0.290637,7807,2269
0,BMD,0.274682,1023,281


## 2. 🧪 Fisher: pathogenic ~ phenotype ⭐


In [18]:
rows = []
for ph in sorted(d['phenotype'].dropna().unique()):
    tmp = d[['phenotype', 'pathogenic']].dropna().copy()
    tmp['is_ph'] = tmp['phenotype'].eq(ph)
    tab, odds, p, ci = fisher_bool(tmp, 'is_ph', 'pathogenic')
    rows.append({'phenotype': ph, 'odds_ratio': odds, 'p_value': p, 'or_ci_low': ci[1], 'or_ci_high': ci[2]})
res = pd.DataFrame(rows).sort_values('p_value')
display(res)


,phenotype,odds_ratio,p_value,or_ci_low,or_ci_high
0,BMD,0.924317,0.304355,0.798977,1.069320
1,DMD,1.081880,0.304355,0.935174,1.251601


## 3. 📊 phenotype vs mutation_type stacked


In [19]:
tmp = d[['phenotype', 'mutation_type']].dropna()
fig = px.histogram(tmp, x='phenotype', color='mutation_type', barmode='stack', title='Phenotype vs mutation_type')
fig.show()


## 4. 🧪 χ² phenotype ~ mutation_type


In [20]:
tab, chi2, p, dof = chi2_table(d, 'phenotype', 'mutation_type')
display(tab)
display(pd.Series({'chi2': chi2, 'dof': dof, 'p_value': p}).to_frame('value'))


mutation_type,frameshift,large_deletion,missense,nonsense,splice
phenotype,,,,,
BMD,53,26,423,116,68
DMD,525,1051,2344,551,706


,value
chi2,1.552106e+02
dof,4.000000e+00
p_value,1.555584e-32


## 5. 📊 phenotype vs frame_status


In [21]:
tmp = d[['phenotype', 'frame']].dropna()
fig = px.histogram(tmp, x='phenotype', color='frame', barmode='stack', title='Phenotype vs frame_status')
fig.show()


## 6. 🧪 χ² phenotype ~ frame_status ⭐


In [22]:
tab, chi2, p, dof = chi2_table(d, 'phenotype', 'frame')
display(tab)
display(pd.Series({'chi2': chi2, 'dof': dof, 'p_value': p}).to_frame('value'))


frame,in-frame,out-of-frame
phenotype,,
BMD,423,263
DMD,2344,2833


,value
chi2,6.459485e+01
dof,1.000000e+00
p_value,9.199566e-16


## 7. 📊 phenotype vs domain (top domains)


In [23]:
top_domains = d['domain_clean'].value_counts().head(12).index
tmp = d[d['domain_clean'].isin(top_domains)][['phenotype', 'domain_clean']]
fig = px.histogram(tmp, x='phenotype', color='domain_clean', barmode='stack', title='Phenotype vs top domains')
fig.show()


## 8. 🧪 χ² phenotype ~ domain


In [24]:
tmp = d[d['domain_clean'].isin(d['domain_clean'].value_counts().head(20).index)]
tab, chi2, p, dof = chi2_table(tmp, 'phenotype', 'domain_clean')
display(tab)
display(pd.Series({'chi2': chi2, 'dof': dof, 'p_value': p}).to_frame('value'))


domain_clean,Calponin-homology (CH1) 1,Calponin-homology (CH2) 2,Interaction with SYNM,Spectrin 1,Spectrin 11,Spectrin 12,Spectrin 13,Spectrin 14,Spectrin 15,Spectrin 16,Spectrin 2,Spectrin 22,Spectrin 23,Spectrin 3,Spectrin 4,Spectrin 5,Spectrin 6,Spectrin 7,Spectrin 8,Spectrin 9
phenotype,,,,,,,,,,,,,,,,,,,,
BMD,10,14,26,7,8,15,11,6,12,6,17,8,6,14,12,10,7,9,6,10
DMD,87,105,178,104,82,90,87,95,86,84,109,69,93,94,116,121,106,98,70,86


,value
chi2,17.802861
dof,19.000000
p_value,0.535641


## 9. 📊 phenotype vs interval_length


In [25]:
tmp = d[['phenotype', 'interval_length_num']].dropna()
fig = px.box(tmp, x='phenotype', y='interval_length_num', points='outliers', title='Phenotype vs interval_length')
fig.update_yaxes(type='log')
fig.show()


## 10. 🧪 Kruskal phenotype vs interval_length


In [26]:
names, groups, stat, p = kruskal_group(d, 'phenotype', 'interval_length_num')
out = pd.Series({'groups': ', '.join([str(x) for x in names]), 'kruskal_h': stat, 'p_value': p})
display(out.to_frame('value'))


,value
groups,"BMD, DMD"
kruskal_h,47.212415
p_value,0.0


## 11. 📊 phenotype vs REVEL


In [27]:
tmp = d[['phenotype', 'revel_num']].dropna()
fig = px.box(tmp, x='phenotype', y='revel_num', points='outliers', title='Phenotype vs REVEL')
fig.show()


## 12. 🧪 Kruskal phenotype vs REVEL


In [28]:
names, groups, stat, p = kruskal_group(d, 'phenotype', 'revel_num')
out = pd.Series({'groups': ', '.join([str(x) for x in names]), 'kruskal_h': stat, 'p_value': p})
display(out.to_frame('value'))


,value
groups,"BMD, DMD"
kruskal_h,4.217865
p_value,0.04
